# FMI — score every configured brand × peer

Run after `01_extract.ipynb`, with the starter .venv kernel. No database calls.

Each row of `inputs/peers.csv` produces one score for that brand and that peer.
Three peers for a brand means three score rows, not a pooled peer median.
Social data stays the same for a brand; commercial/trend and the final score can vary.

Default is a **provisional real-data preview**. Reporting scope, units, source
quarter allocations, selling-season alignment and social coverage still need review.
Use `MODE="reviewed"` for manually annotated financial inputs and review checks.

See [Metrics and methodology](../METRICS.md) for formulas and assumptions.

In [ ]:
import runpy
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
project_root = next(
    p
    for p in [cwd, *cwd.parents]
    if (p / "pyproject.toml").is_file() and (p / "src/aily_project").is_dir()
)
folder = project_root / "scripts/fashion-momentum-index"
assert Path(sys.prefix).resolve() == (project_root / ".venv").resolve(), (
    "Select the starter project's .venv kernel, restart, then Run All."
)
fmi = runpy.run_path(str(folder / "fmi.py"))
raw_dir = folder / "data/raw"
processed_dir = folder / "data/processed"
results_dir = folder / "results"
for directory in (processed_dir, results_dir):
    directory.mkdir(parents=True, exist_ok=True)

MODE = "preview"  # preview (real downloads), reviewed, or demo
assert MODE in {"preview", "reviewed", "demo"}
# Brand list and peers: inputs/peers.csv (one comparison per row).
# Show/social settings: inputs/analyses.csv. Source periods: inputs/entities.csv.

In [ ]:
if MODE == "demo":
    financials, posts, labels, analysis = fmi["demo_inputs"]()
    analyses = [analysis]  # Original synthetic two-peer-median regression example.
else:
    settings = pd.read_csv(folder / "inputs/analyses.csv").fillna("")
    peer_config = pd.read_csv(folder / "inputs/peers.csv").fillna("")
    entities = pd.read_csv(folder / "inputs/entities.csv").fillna("")
    analyses = fmi["comparison_analyses"](settings, peer_config)
    display(pd.DataFrame(analyses)[["brand", "financial_entity", "peer_entity"]])
    required = ["revenue_actuals.csv", "instagram_posts.csv"]
    missing = [name for name in required if not (raw_dir / name).is_file()]
    assert not missing, f"Run notebook 01 first. Missing CSVs: {missing}"
    actuals = pd.read_csv(raw_dir / "revenue_actuals.csv")
    forecast_path = raw_dir / "revenue_forecasts.csv"
    forecasts = (
        pd.read_csv(forecast_path)
        if forecast_path.is_file()
        else pd.DataFrame(columns=actuals.columns)
    )
    posts = pd.read_csv(raw_dir / "instagram_posts.csv", dtype={"post_key": str})
    review_path = processed_dir / "collection_review.csv"
    labels = (
        pd.read_csv(review_path, dtype={"post_key": str, "collection_related": str})
        if review_path.is_file()
        else pd.DataFrame(columns=["handle", "post_key", "collection_related"])
    )
    print(f"{len(analyses)} configured comparisons; {len(posts)} social posts.")

if MODE == "reviewed":
    external = pd.read_csv(folder / "inputs/external_financials.csv")
    frames = [frame for frame in (actuals, forecasts, external) if not frame.empty]
    raw_financials = pd.concat(frames, ignore_index=True)
    selections = pd.read_csv(
        folder / "inputs/financial_selection.csv", dtype=str
    ).fillna("")
    annotations = pd.read_csv(folder / "inputs/financial_annotations.csv", dtype=str)
    financials = fmi["select_financials"](raw_financials, selections, annotations)
    financials.to_csv(processed_dir / "selected_financials.csv", index=False)

In [ ]:
if MODE == "preview":
    # Prepare each entity independently so one absent peer does not stop other pairs.
    needed_entities = sorted(
        {
            name
            for analysis in analyses
            for name in (analysis["financial_entity"], analysis["peer_entity"])
        }
    )
    cutoff = max(pd.Timestamp(analysis["as_of"]) for analysis in analyses).isoformat()
    prepared = []
    preparation_issues = {}
    for entity in needed_entities:
        try:
            choices = fmi["entity_period_choices"](entities, [entity])
            prepared.append(fmi["preview_periods"](actuals, choices, cutoff))
        except (ValueError, KeyError, TypeError) as error:
            preparation_issues[entity] = str(error)
    financials = (
        pd.concat(prepared, ignore_index=True)
        if prepared
        else actuals.iloc[:0].assign(period_role="")
    )
    financials.to_csv(processed_dir / "preview_financials.csv", index=False)
    display(
        financials[
            [
                "entity",
                "period_role",
                "year",
                "quarter",
                "reported_date",
                "amount",
                "source_row_id",
            ]
        ]
    )
    for entity, issue in preparation_issues.items():
        print(f"{entity}: {issue}")

In [ ]:
labels_by_entity = {}
if MODE == "preview":
    review_frames = []
    social_preparation_issues = {}
    for analysis in settings.to_dict("records"):
        entity = analysis["financial_entity"]
        if entity not in peer_config["financial_entity"].values:
            continue
        brand_posts = posts.loc[
            posts["handle"].str.lower().eq(analysis["handle"].lower())
        ]
        try:
            if analysis["route"] == "show":
                suggestions = fmi["preview_collection_labels"](
                    brand_posts,
                    labels,
                    analysis["collection"],
                    analysis["other_collection"],
                )
                local_dates = (
                    pd.to_datetime(
                        brand_posts["date_posted"],
                        utc=True,
                        errors="coerce",
                        format="mixed",
                    )
                    .dt.tz_convert(analysis["timezone"])
                    .dt.date
                )
                show = pd.Timestamp(analysis["show_date"]).date()
                window = local_dates.between(show, show + pd.Timedelta(days=7))
                review = brand_posts.loc[
                    window, ["handle", "post_key", "date_posted", "url", "description"]
                ].merge(
                    suggestions[
                        ["handle", "post_key", "collection_related", "label_source"]
                    ],
                    on=["handle", "post_key"],
                    validate="one_to_one",
                )
                review["financial_entity"] = entity
                review_frames.append(review)
                labels_by_entity[entity] = suggestions
            else:
                labels_by_entity[entity] = labels
        except (ValueError, KeyError, TypeError) as error:
            social_preparation_issues[entity] = str(error)
            labels_by_entity[entity] = labels
    review_output = (
        pd.concat(review_frames, ignore_index=True)
        if review_frames
        else pd.DataFrame(columns=["handle", "post_key", "collection_related"])
    )
    review_output.to_csv(processed_dir / "preview_collection_review.csv", index=False)
    display(review_output)
    # All unknown show-week labels still make that brand's social component unavailable.

In [ ]:
# Per pair: relative = brand growth - configured peer growth, not a peer pool.
# IR = clip(50 + 5*relative_pp); trend compares latest and earlier relative growth.
# FMI = 0.50*IR + 0.30*social + 0.20*trend, with no missing-value reweighting.
rows = []
for analysis in analyses:
    pair_analysis = dict(analysis)
    pair_labels = labels
    if MODE == "preview":
        pair_analysis["point_in_time"] = False  # Current-snapshot proxy only.
        pair_labels = labels_by_entity.get(analysis["financial_entity"], labels)
        if analysis["financial_entity"] in social_preparation_issues:
            pair_analysis["social_preparation_issue"] = social_preparation_issues[
                analysis["financial_entity"]
            ]
        pair_analysis["preparation_gaps"] = " | ".join(
            preparation_issues[name]
            for name in (analysis["financial_entity"], analysis["peer_entity"])
            if name in preparation_issues
        )
        if pair_analysis.get("social_preparation_issue"):
            pair_analysis["preparation_gaps"] += (
                " | social preparation: " + pair_analysis["social_preparation_issue"]
            )
    rows.append(
        fmi["score_analysis"](
            financials, posts, pair_labels, pair_analysis, preview=MODE == "preview"
        )
    )
scores = pd.DataFrame(rows)
result_name = {
    "preview": "provisional_fmi_scores.csv",
    "reviewed": "fmi_scores.csv",
    "demo": "demo_fmi_scores.csv",
}[MODE]
scores.to_csv(results_dir / result_name, index=False)
columns = [
    "brand",
    "peer_entity",
    "brand_growth_pct",
    "peer_growth_pct",
    "ir_score",
    "social_score",
    "trend_score",
    "fmi",
    "classification",
    "gaps",
]
display(scores.reindex(columns=columns).round(2))
print(f"Saved {len(scores)} comparison rows: {results_dir / result_name}")

In [ ]:
if MODE == "preview":
    # One illustrative scenario per scored pair; no guessed currency metadata.
    scenario_rows = []
    evidence = []
    source = pd.concat(
        [frame for frame in (actuals, forecasts) if not frame.empty],
        ignore_index=True,
    )
    for result in scores.to_dict("records"):
        scenario = {
            "brand": result["brand"],
            "financial_entity": result["financial_entity"],
            "peer_entity": result["peer_entity"],
            "comparison_mode": "single-peer",
            "fmi": result["fmi"],
            "status": "Unavailable",
        }
        try:
            if pd.isna(result["fmi"]):
                raise ValueError("FMI unavailable for this brand-peer comparison.")
            choices = fmi["entity_period_choices"](
                entities, [result["financial_entity"]], ("target", "last_year_target")
            )
            target_rows = fmi["preview_periods"](source, choices, result["as_of"])
            amounts = target_rows.set_index("period_role")["amount"]
            scenario.update(
                fmi["sales_scenario"](
                    result["fmi"],
                    float(amounts["target"]),
                    float(amounts["last_year_target"]),
                )
            )
            scenario.update(
                {
                    "status": "PROVISIONAL planning scenario — unverified source units",
                    "source_units": "Light raw units (currency/unit metadata unknown)",
                    "target_fiscal_year": int(
                        choices.loc[choices["period_role"].eq("target"), "year"].iloc[0]
                    ),
                    "target_quarter": choices.loc[
                        choices["period_role"].eq("target"), "quarter"
                    ].iloc[0],
                }
            )
            evidence.append(target_rows)
        except (ValueError, KeyError, TypeError) as error:
            scenario["reason"] = str(error)
        scenario_rows.append(scenario)
    scenarios = pd.DataFrame(scenario_rows)
    scenarios.to_csv(results_dir / "provisional_sales_scenario.csv", index=False)
    target_evidence = (
        pd.concat(evidence, ignore_index=True).drop_duplicates()
        if evidence
        else source.iloc[:0].assign(period_role="")
    )
    target_evidence.to_csv(processed_dir / "preview_target_financials.csv", index=False)
    display(scenarios)

## Configuration and core logic

- `inputs/peers.csv`: one row per brand/financial entity and peer. Every row produces
  one score. Adding three peers produces three comparisons, not combinations of peer groups.
- `inputs/analyses.csv`: one row per scored brand; handle, show date, collection and
  excluded collection captions, timezone, social windows, cutoff and review flags.
  Legacy `peer_1/peer_2` columns are ignored for real runs; peers.csv owns peer selection.
- `inputs/entities.csv`: one row per brand or peer; Light company code and explicit
  oldest/earlier/latest source periods, optional target and prior-year target periods.
  The extraction notebook downloads the union of brands and peers and brand social handles.
- Commercial growth is `(current/prior - 1)*100`. For each pair, subtract only
  that peer's growth. IR = `clip(50+5*relative_pp)`.
- Social uses the brand's show-day..+7 collection median versus all brand posts
  in -30..-1 and +8..+37. It is the same across peers for that brand.
  Caption suggestions and existing manual overrides stay separate from your reviews.
- Trend applies the unchanged ordered >+2 / <-2 / sustained +5 pp rules to the
  pair's relative growth history. FMI weights remain 50% / 30% / 20%.
- Missing peer data affects only that pair; missing brand/social data affects
  that brand's pairs. Results retain unavailable rows and explain `gaps`.
- Preview remains a current-snapshot source-period proxy, not a validated
  collection FMI. No review flags are automatically turned on.
- Scenario baseline is per brand, but each pair's score can yield a different
  planning uplift. These are alternative comparisons, never additive sales effects.
- Demo keeps the original two-peer-median handover example; real CSV runs are
  the requested pairwise extension. See [METRICS.md](../METRICS.md).

Clear outputs before committing. No database writes, scheduler or deployment.